In [ ]:
import os, json, time, hashlib
from pathlib import Path
from typing import TypedDict, List
import pandas as pd
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from pinecone import Pinecone

load_dotenv()
CSV_PATH='data.csv'
STATE_FILE='ingestion_state.json'
INDEX_NAME=os.getenv('PINECONE_INDEX','rag-index')
NAMESPACE='default'

emb=GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')
llm=ChatGoogleGenerativeAI(model='gemini-1.5-flash',temperature=0)
pc=Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index=pc.Index(INDEX_NAME)
vs=PineconeVectorStore(index=index, embedding=emb, namespace=NAMESPACE)
splitter=RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)

IOT_WORDS={'iot','sensor','firmware','device','mqtt','zigbee','network','security'}

def row_hash(row): return hashlib.md5(json.dumps(row,sort_keys=True).encode()).hexdigest()

def load_state():
    return json.loads(Path(STATE_FILE).read_text()) if Path(STATE_FILE).exists() else {}

def save_state(s): Path(STATE_FILE).write_text(json.dumps(s,indent=2))

def ingest_csv_incremental():
    df=pd.read_csv(CSV_PATH)
    state=load_state(); new_docs=[]; new_state={}
    for i,row in df.fillna('').iterrows():
        data=row.to_dict(); h=row_hash(data); rid=str(i)
        new_state[rid]=h
        if state.get(rid)==h: continue
        text=' | '.join(f'{k}: {v}' for k,v in data.items())
        for j,ch in enumerate(splitter.split_text(text)):
            new_docs.append(Document(page_content=ch, metadata={'row_id':rid,'chunk':j}))
    if new_docs:
        vs.add_documents(new_docs)
        print(f'Upserted {len(new_docs)} chunks')
    else:
        print('No changes found')
    save_state(new_state)

class S(TypedDict):
    query:str; route:str; docs:List[Document]; answer:str

def classify(s:S):
    q=s['query'].lower(); s['route']='iot' if any(w in q for w in IOT_WORDS) else 'reject'; return s

def retrieve(s:S):
    s['docs']=vs.similarity_search(s['query'],k=4); return s

def generate(s:S):
    ctx='\n\n'.join(d.page_content for d in s['docs'])
    prompt=ChatPromptTemplate.from_template('Answer only from context.\nContext:{context}\nQuestion:{question}')
    s['answer']=(prompt|llm).invoke({'context':ctx,'question':s['query']}).content; return s

def reject(s:S): s['answer']='Query not related to IoT domain'; return s

g=StateGraph(S)
g.add_node('classify',classify); g.add_node('retrieve',retrieve); g.add_node('generate',generate); g.add_node('reject',reject)
g.set_entry_point('classify')
g.add_conditional_edges('classify', lambda s:s['route'], {'iot':'retrieve','reject':'reject'})
g.add_edge('retrieve','generate'); g.add_edge('generate',END); g.add_edge('reject',END)
app=g.compile()

def watch_csv(interval=120):
    last=0
    while True:
        m=Path(CSV_PATH).stat().st_mtime
        if m!=last:
            ingest_csv_incremental(); last=m
        time.sleep(interval)

def chat():
    while True:
        q=input('You: ')
        if q.lower() in {'exit','quit'}: break
        out=app.invoke({'query':q,'route':'','docs':[],'answer':''})
        print('Bot:',out['answer'])

if __name__=='__main__':
    import threading
    threading.Thread(target=watch_csv,daemon=True).start()
    ingest_csv_incremental()
    chat()
